# Trabalho 1: Redução de Dimensionalidade — Reconhecimento de Padrões

**Alunos:** Alan Saar & Leonardo Herkenhoff  
**Professor:** Prof. Dr. Sérgio Nery Simões  
**Programa:** PPCOMP - IFES | 2026

Este notebook foi desenvolvido com fins didáticos e práticos para que você possa entender o funcionamento dos algoritmos de redução de dimensionalidade (**PCA**, **t-SNE** e **UMAP**), avaliar qualitativa e quantitativamente as projeções e estudar o comportamento de seus hiperparâmetros no dataset **MNIST**.

---

## 1. Configurando o Ambiente e Importando Funções

Vamos adicionar a pasta raiz ao caminho do sistema para que possamos importar os módulos criados na pasta `scripts/` de forma limpa e modular.

In [ ]:
import sys
import os

# Adicionar diretório raiz do projeto ao sys.path para importações
sys.path.append(os.path.abspath(os.path.join("..")))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Importar as funções desenvolvidas em scripts/run_experiments.py
from scripts.run_experiments import (
    load_and_preprocess_mnist,
    run_pca,
    run_tsne,
    run_umap,
    evaluate_embedding,
    plot_embedding,
    run_stability_experiment,
    run_hyperparameter_sweep,
    run_downstream_evaluation
)

print("[ OK ] Funções importadas com sucesso do pipeline de scripts!")

## 2. Carregando e Pré-processando o MNIST

O MNIST original contém 70.000 imagens de tamanho $28 \times 28$ pixels (784 dimensões). 
Para fins acadêmicos e para garantir uma execução rápida diretamente no seu notebook local (sem travar a máquina e rodando em segundos), utilizaremos uma **amostragem estratificada** de **3.000 instâncias**. A estratificação garante que a proporção original das classes (dígitos de 0 a 9) seja mantida exata.

**Pré-processamento aplicado:**
1. Amostragem de 3.000 amostras estratificadas.
2. Normalização (divisão por 255.0) para projetar os pixels no intervalo $[0, 1]$.

In [ ]:
# Carrega e normaliza uma amostra representativa de 3000 dígitos do MNIST
X, y = load_and_preprocess_mnist(n_samples=3000, random_state=42)

--- 
## 3. Parte A — Fidelidade Estrutural (PCA vs. t-SNE vs. UMAP)

Agora vamos executar a redução de dimensionalidade para as 2 dimensões visuais usando os três métodos. 

### 3.1 PCA (Principal Component Analysis)
*   **Tipo:** Linear e Determinístico.
*   **Conceito:** Encontra as direções de máxima variância nos dados e projeta de forma linear.

In [ ]:
print("[*] Executando PCA...")
pca_emb, pca_dur = run_pca(X)

# Avaliar quantitativamente as métricas do PCA
pca_metrics = evaluate_embedding(X, pca_emb, y, pca_dur, "PCA")

# Plotar e salvar a projeção
plot_embedding(pca_emb, y, "PCA - Projeção 2D (MNIST)", "projection_pca.png")

# Visualizar aqui no notebook
plt.figure(figsize=(8, 6))
plt.scatter(pca_emb[:, 0], pca_emb[:, 1], c=y, cmap='tab10', s=4, alpha=0.7)
plt.colorbar(label='Dígitos')
plt.title("PCA - Projeção 2D")
plt.show()

### 3.2 t-SNE (t-distributed Stochastic Neighbor Embedding)
*   **Tipo:** Não linear e Probabilístico (Estocástico).
*   **Conceito:** Converte distâncias euclidianas entre pontos em probabilidades condicionais de vizinhança. Utiliza a distribuição t-Student para mitigar o *crowding problem* no espaço projetado. Preserva fortemente as vizinhanças locais.

In [ ]:
print("[*] Executando t-SNE (com redução preliminar PCA-50 recomendada)...")
tsne_emb, tsne_dur = run_tsne(X, random_state=42)

# Avaliar quantitativamente as métricas do t-SNE
tsne_metrics = evaluate_embedding(X, tsne_emb, y, tsne_dur, "t-SNE")

# Plotar e salvar a projeção
plot_embedding(tsne_emb, y, "t-SNE - Projeção 2D (MNIST)", "projection_tsne.png")

# Visualizar aqui no notebook
plt.figure(figsize=(8, 6))
plt.scatter(tsne_emb[:, 0], tsne_emb[:, 1], c=y, cmap='tab10', s=4, alpha=0.7)
plt.colorbar(label='Dígitos')
plt.title("t-SNE - Projeção 2D")
plt.show()

### 3.3 UMAP (Uniform Manifold Approximation and Projection)
*   **Tipo:** Não linear e Estocástico baseado em Topologia Algébrica.
*   **Conceito:** Supõe que os dados estão distribuídos em um manifold de baixa dimensão e busca preservar a estrutura difusa local e global por meio de otimização de entropia cruzada difusa.

In [ ]:
print("[*] Executando UMAP...")
umap_emb, umap_dur = run_umap(X, random_state=42)

# Avaliar quantitativamente as métricas do UMAP
umap_metrics = evaluate_embedding(X, umap_emb, y, umap_dur, "UMAP")

# Plotar e salvar a projeção
plot_embedding(umap_emb, y, "UMAP - Projeção 2D (MNIST)", "projection_umap.png")

# Visualizar aqui no notebook
plt.figure(figsize=(8, 6))
plt.scatter(umap_emb[:, 0], umap_emb[:, 1], c=y, cmap='tab10', s=4, alpha=0.7)
plt.colorbar(label='Dígitos')
plt.title("UMAP - Projeção 2D")
plt.show()

### 3.4 Comparativo Quantitativo Geral

Vamos exibir a tabela comparativa contendo:
1.  **Trustworthiness (Confiabilidade):** Evita falsas vizinhanças (falsos positivos).
2.  **Continuity (Continuidade):** Evita quebras de vizinhança (falsos negativos).
3.  **Silhouette Score:** Medida de separação e coesão dos clusters baseada nas classes reais do MNIST.
4.  **Tempo de Execução:** Custo de processamento temporal.

In [ ]:
df_metrics = pd.DataFrame([pca_metrics, tsne_metrics, umap_metrics])
print("=== Tabela de Fidelidade Estrutural ===")
display(df_metrics.round(4))

--- 
## 4. Parte B — Estabilidade e Robustez (10 Sementes & Hiperparâmetros)

Algoritmos estocásticos baseiam-se em inicializações aleatórias que podem produzir ótimos locais diferentes. 
Nesta etapa, rodamos os experimentos com **10 sementes aleatórias distintas** e geramos estatísticas robustas (Média e Desvio Padrão).

In [ ]:
# Executa a estabilidade de 10 seeds, salvando boxplot e estatísticas em relatorio/img/
df_stability = run_stability_experiment(X, y, n_seeds=10)

### 4.2 Varredura de Hiperparâmetros

Abaixo geramos a varredura visual e quantitativa dos hiperparâmetros:
-   **t-SNE:** Variando a perplexidade (influencia se focamos em estruturas muito locais ou globais).
-   **UMAP:** Variando `n_neighbors` (controla o tamanho da vizinhança local considerada).

In [ ]:
# Roda a varredura de perplexidade (t-SNE) e vizinhos (UMAP)
run_hyperparameter_sweep(X, y)

--- 
## 5. Parte C — Avaliação Downstream (k-NN Classificação)

**A pergunta científica crucial:** O embedding bidimensional gerado preserva a informação semântica original a ponto de conseguirmos classificar o dígito corretamente com um algoritmo simples?

Vamos rodar um classificador **k-NN ($k=5$)** no:
1.  **Espaço Original (784 dimensões)**
2.  **Espaço PCA (2 dimensões)**
3.  **Espaço t-SNE (2 dimensões)**
4.  **Espaço UMAP (2 dimensões)**

In [ ]:
# Roda a classificação downstream, gerando gráficos de barras e exportando CSV
run_downstream_evaluation(X, y)

### 5.1 O que aprendemos com a Avaliação Downstream?
-   O **PCA 2D** costuma ter acurácia baixa devido à perda massiva de informações por tentar aproximar tudo linearmente.
-   **t-SNE** e **UMAP** conseguem agrupar classes semelhantes em clusters densos e bem separados, o que permite ao k-NN obter acurácias muito próximas à do espaço original de 784D, mesmo reduzindo os dados para meras 2 dimensões!
-   Essa discussão teórica é de altíssimo nível para o seu relatório e comprova cientificamente o poder das projeções não lineares na preservação de estruturas de alta dimensão.